# Newberry 2-D probabilistic example: 150 °C

Released tutorial data are downloaded to `data/`; standard Newberry preprocessing is created there on the first run.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from geopfa.datasets import (
    ensure_newberry_tutorial_processed_data,
    setup_newberry_tutorial_data,
)
from geopfa.prob import ProbabilisticConfig, load_processed_pfa, run_probabilistic
from geopfa.prob.evidence_prior_profiles import evidence_prior_profile_from_dict
from geopfa.prob.evidence_priors import derive_evidence_coefficient_priors


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from a geoPFA checkout")


repo_root = find_repo_root(Path.cwd().resolve())
project_dir = repo_root / "examples" / "Newberry" / "2D"
data_dir = project_dir / "data"
setup_newberry_tutorial_data(data_dir)
processed_config_path, processed_data_dir = ensure_newberry_tutorial_processed_data(
    project_dir, dimensions="2d"
)
pfa, input_artifacts = load_processed_pfa(
    processed_config_path, processed_data_dir, crs="EPSG:26910"
)


In [ ]:
config_dict = {
    "enabled": True,
    "output_dir": "outputs/newberry_2d_150c",
    "dimensions": "2d",
    "alpha": {
        "heat": {
            "mode": "thermal_layer_exceedance",
            "layer": "temperature_model_500m",
            "threshold": 150.0,
            "force_prior_predictive": True,
            "use_evidence_prior": False,
        },
        "reservoir": {
            "mode": "scalar",
            "scalar_fallback_pr0": 0.5,
            "force_prior_predictive": True,
            "use_evidence_prior": True,
        },
        "insulation": {
            "mode": "scalar",
            "scalar_fallback_pr0": 0.5,
            "force_prior_predictive": True,
            "use_evidence_prior": True,
        },
    },
    "evidence": {
        "standardization": "prediction_support",
        "include_layers": [
            "mt_resistivity_joint_inv",
            "temperature_model_500m",
            "earthquakes",
            "lineaments",
        ],
        "regularization": {},
    },
    "spatial_field": {"enabled": False},
    "inference": {
        "backend": "gblk",
        "gblk_bayesian": {
            "enabled": True,
            "n_draws": 512,
            "seed": 20260930,
            "ci_level": 0.95,
            "cluster_effect": False,
        },
    },
    "calibration": {"method": "none"},
    "combination": {"rule": "product"},
    "scenarios": [],
    "outputs": {
        "posterior_draw_blocks": False,
        "posterior_draw_block_size": 32,
        "format": ["parquet"],
    },
}

profile_path = project_dir / "config" / "newberry_2d_150c_priors.json"
profile = evidence_prior_profile_from_dict(json.loads(profile_path.read_text()))
resolved_priors = derive_evidence_coefficient_priors(
    profile,
    {
        "reservoir": (
            "mt_resistivity_joint_inv",
            "earthquakes",
            "lineaments",
        ),
        "insulation": (
            "mt_resistivity_joint_inv",
            "earthquakes",
            "temperature_model_500m",
        ),
    },
)
config_dict["evidence"]["regularization"] = {
    "prior_means": dict(resolved_priors.prior_means),
    "prior_precisions": dict(resolved_priors.prior_precisions),
    "fixed_coefficients": dict(resolved_priors.fixed_coefficients),
}
config_dict["output_dir"] = str(project_dir / "outputs" / "newberry_2d_150c")
config = ProbabilisticConfig.from_dict(config_dict)
config.validate_raise()


In [ ]:
model_result = run_probabilistic(
    pfa,
    config,
    input_artifacts=input_artifacts,
)

surfaces = {
    **{name: result.probability for name, result in model_result.components.items()},
    "combined": model_result.combined,
}
summary = pd.DataFrame(
    {
        name: surface["probability"].describe()[["mean", "std", "min", "max"]]
        for name, surface in surfaces.items()
    }
).T
summary


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), layout="constrained")
for axis, (name, surface) in zip(axes, surfaces.items(), strict=True):
    surface.plot(
        ax=axis,
        column="probability",
        cmap="magma",
        vmin=0.0,
        vmax=1.0,
        markersize=1,
        legend=True,
    )
    axis.set_title(name)
    axis.set_axis_off()
plt.show()
